# 4. Feature engineering

Run `python -m pip install -e ".[dev]"` from the repository root first. Each notebook runs independently and imports the same tested Python functions. Source timestamps retain their original local calendar meaning. Set `RETAIL_ETL_INPUT` to use another source and adjust `AS_OF` to its reporting cutoff.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from retail_etl.pipeline import project_root, read_source
from retail_etl.transform import prepare_data, feature_engineering, validate_output

ROOT = project_root()
SOURCE = Path(os.getenv("RETAIL_ETL_INPUT", str(ROOT / "data/raw/Online Retail.xlsx")))
OUTPUT = Path(os.getenv("RETAIL_ETL_OUTPUT", str(ROOT / "outputs")))
AS_OF = "2011-12-09"  # Inclusive end date of the bundled historical dataset.


## Always start from the shared cleaned dataset

This replaces the old path that engineered raw, uncleaned records and overwrote the automation output.

In [ ]:
prepared = prepare_data(read_source(SOURCE), as_of=AS_OF)
final = feature_engineering(prepared.cleaned)
validate_output(final)
final.head()

## Canonical features

The output contains 20 columns: eight source fields, eleven analytical fields, and one source row number. The same ordered contract generates MySQL DDL and insert statements.

In [ ]:
from retail_etl.schema import COLUMN_TYPES
pd.Series(COLUMN_TYPES, name="MySQL definition")

## Invoice totals and average order value

An invoice total is a sum across its lines. Average order value is the mean across distinct invoice totals, not a line-weighted mean of repeated invoice totals. This exploration does not add a repeated AOV column to the export.

In [ ]:
invoice_totals = final.groupby("invoice_no")["total_amount"].sum()
average_order_value = sum(invoice_totals) / len(invoice_totals)
print("Invoice count:", len(invoice_totals))
print("Average order value (GBP):", round(average_order_value, 2))

## Customer frequency

These labels describe the entire observed period, not whether a customer was new at the time of a particular purchase.

In [ ]:
orders_per_customer = final.groupby("customer_id")["invoice_no"].nunique()
orders_per_customer.describe()